In [ ]:

Ce notebook charge la liste `quizzes_data` du script Python, valide sa structure, analyse la composition des quizzes, exporte les données et inspecte la distribution des thèmes et niveaux.

In [ ]:
import csv
import importlib.util
import json
import os
from collections import Counter
from pathlib import Path

NOTEBOOK_DIR = Path(__file__).resolve().parent
SCRIPT_PATH = NOTEBOOK_DIR.parent / "tools" / "quiz" / "enrichment" / "generator" / "generate_2nde_francais.py"

spec = importlib.util.spec_from_file_location("generate_2nde_francais", SCRIPT_PATH)
quiz_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(quiz_module)

raw_quizzes_data = getattr(quiz_module, "quizzes_data", [])

print(f"Chargé {len(raw_quizzes_data)} quiz(x) depuis {SCRIPT_PATH}")


Conversion de la structure de tuples en une liste de dictionnaires pour faciliter l'analyse. Chaque quiz contient : `id`, `title`, `subject`, `level` et `questions`.

In [ ]:
quizzes = []

for item in raw_quizzes_data:
    if not isinstance(item, (list, tuple)) or len(item) != 5:
        raise ValueError(f"Quiz invalide : {item}")
    qid, title, subject, level, questions = item
    quizzes.append({
        "id": qid,
        "title": title,
        "subject": subject,
        "level": level,
        "questions": list(questions) if isinstance(questions, (list, tuple)) else [],
    })

print(f"Converti en {len(quizzes)} dict(s) de quiz")


Vérification des clés attendues, des types de question validés (`qcm`, `vrai-faux`) et de l'unicité des identifiants de question.

In [ ]:
EXPECTED_QUESTION_KEYS = {"id", "type", "question"}
VALID_TYPES = {"qcm", "vrai-faux"}

errors = []
question_ids = []

for quiz in quizzes:
    if not isinstance(quiz["id"], int):
        errors.append(f"Quiz id non entier : {quiz}")
    if not isinstance(quiz["questions"], list):
        errors.append(f"Questions non list pour quiz {quiz['id']}")
    for question in quiz["questions"]:
        if not isinstance(question, dict):
            errors.append(f"Question non dict dans quiz {quiz['id']}: {question}")
            continue
        missing = EXPECTED_QUESTION_KEYS - question.keys()
        if missing:
            errors.append(f"Question manquante de clés {missing} dans quiz {quiz['id']} question {question.get('id')}")
        qtype = str(question.get("type", "")).strip().lower().replace("_", "-")
        if qtype not in VALID_TYPES:
            errors.append(f"Type invalide {qtype} dans question {question.get('id')} de quiz {quiz['id']}")
        question_ids.append(question.get("id"))
        if qtype == "qcm":
            if not question.get("options") or not isinstance(question.get("options"), list):
                errors.append(f"Options manquantes ou invalides pour {question.get('id')}")
            if question.get("correct_option") not in question.get("options", []):
                errors.append(f"Correct_option invalide pour {question.get('id')}")
        elif qtype == "vrai-faux":
            if "correct" not in question:
                errors.append(f"Champ correct manquant pour vrai-faux {question.get('id')}")

duplicate_ids = [item for item, count in Counter(question_ids).items() if count > 1]
if duplicate_ids:
    errors.append(f"Identifiants de question dupliqués : {duplicate_ids}")

print("Erreurs de validation :")
for line in errors[:20]:
    print("-", line)

print("\nNombre total d'erreurs :", len(errors))


Calcul du nombre de questions par type (`qcm`, `vrai-faux`) pour chaque quiz, et repérage des déséquilibres éventuels.

In [ ]:
quiz_type_counts = []

for quiz in quizzes:
    type_counter = Counter()
    for question in quiz["questions"]:
        qtype = str(question.get("type", "")).strip().lower().replace("_", "-")
        if qtype not in VALID_TYPES:
            qtype = "invalide"
        type_counter[qtype] += 1
    quiz_type_counts.append({
        "quiz_id": quiz["id"],
        "title": quiz["title"],
        "total_questions": len(quiz["questions"]),
        "qcm": type_counter["qcm"],
        "vrai-faux": type_counter["vrai-faux"],
        "invalides": type_counter["invalide"],
    })

for summary in quiz_type_counts[:10]:
    print(summary)

imbalances = [summary for summary in quiz_type_counts if summary["qcm"] == 0 or summary["vrai-faux"] == 0]
print(f"\nQuizzes avec un type unique ou manquant : {len(imbalances)}")


Écriture des données nettoyées vers `audit_quizzes.json`, `audit_quizzes.csv` et `audit_questions.csv`.

In [ ]:
EXPORT_DIR = NOTEBOOK_DIR / "export"
EXPORT_DIR.mkdir(exist_ok=True)

json_path = EXPORT_DIR / "audit_quizzes.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(quizzes, f, ensure_ascii=False, indent=2)

csv_quiz_path = EXPORT_DIR / "audit_quizzes.csv"
csv_question_path = EXPORT_DIR / "audit_questions.csv"

with open(csv_quiz_path, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["quiz_id", "title", "subject", "level", "question_count"])
    writer.writeheader()
    for quiz in quizzes:
        writer.writerow({
            "quiz_id": quiz["id"],
            "title": quiz["title"],
            "subject": quiz["subject"],
            "level": quiz["level"],
            "question_count": len(quiz["questions"]),
        })

with open(csv_question_path, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["quiz_id", "question_id", "type", "question", "correct_answer", "options", "explanation"])
    writer.writeheader()
    for quiz in quizzes:
        for question in quiz["questions"]:
            writer.writerow({
                "quiz_id": quiz["id"],
                "question_id": question.get("id"),
                "type": str(question.get("type", "")).strip().lower().replace("_", "-"),
                "question": question.get("question"),
                "correct_answer": question.get("correct_option") if str(question.get("type", "")).strip().lower().replace("_", "-") == "qcm" else question.get("correct"),
                "options": json.dumps(question.get("options", []), ensure_ascii=False),
                "explanation": question.get("explanation"),
            })

print(f"Export JSON créé : {json_path}")
print(f"Export CSV quiz : {csv_quiz_path}")
print(f"Export CSV questions : {csv_question_path}")


Distribution des matières, niveaux et titres de quiz pour comprendre la répartition du corpus.

In [ ]:
subject_counter = Counter(quiz["subject"] for quiz in quizzes)
level_counter = Counter(quiz["level"] for quiz in quizzes)

title_words = Counter()
for quiz in quizzes:
    for word in str(quiz["title"]).split():
        title_words[word.strip("-_:;,.?\"'()[]")[:30].lower()] += 1

print("Distribution des matières :")
for subject, count in subject_counter.items():
    print(f"- {subject}: {count}")

print("\nDistribution des niveaux :")
for level, count in level_counter.items():
    print(f"- {level}: {count}")

print("\nMots les plus fréquents dans les titres :")
for word, count in title_words.most_common(20):
    print(f"- {word}: {count}")